In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r'D:\Movie_Sentiment_Analysis\data\raw\IMDB.csv')

In [4]:
def clean_data(df):
    # Replace all instances of "positive" with "1" in column: 'sentiment'
    df['sentiment'] = df['sentiment'].str.replace(
        "positive", "1", case=False, regex=False)
    # Replace all instances of "negative" with "0" in column: 'sentiment'
    df['sentiment'] = df['sentiment'].str.replace(
        "negative", "0", case=False, regex=False)
    return df


df_clean = clean_data(df.copy())
df = df_clean.copy()
df.head(8)

,review,sentiment
0,Film version of Sandra Bernhard's one-woman of...,0
1,I switched this on (from cable) on a whim and ...,1
2,The `plot' of this film contains a few holes y...,0
3,"Some amusing humor, some that falls flat, some...",0
4,What can you say about this movie? It was not ...,0
5,So we saw this on DVD at our apartment here in...,1
6,This is a good example a film that in spite of...,1
7,Awesome Movie! Great combination of talents! I...,1


In [5]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    # Handle missing values
    if text is None:
        return ""

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # Join back into a sentence
    return " ".join(tokens)

In [6]:
df['review'].apply(preprocess_text)

0      film version sandra bernhards onewoman offbroa...
1      switched cable whim treated quite surprisealth...
2      plot film contains hole could drive massive tr...
3      amusing humor fall flat decent acting quite at...
4      say movie terrible good two day earlier watche...
                             ...                        
995    exactly new story line romantic comedy make co...
996    first saw movie younger child sister told thou...
997    people stated th season south park started tre...
998    nothing director juvenile fantasy come life mo...
999    spoiler alert throughout australia summer turn...
Name: review, Length: 1000, dtype: object

In [7]:
sentences = df["review"].apply(lambda x: x.split())

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1, 2),
    stop_words='english'
)

X = vectorizer.fit_transform(df["review"])

y = df["sentiment"].astype(int)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    print("=" * 60)
    print(name)

    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))

    if y_prob is not None:
        print("ROC AUC  :", roc_auc_score(y_test, y_prob))

Logistic Regression
Accuracy : 0.71
Precision: 0.693069306930693
Recall   : 0.7216494845360825
F1 Score : 0.7070707070707071
ROC AUC  : 0.7761985787208487
Naive Bayes
Accuracy : 0.705
Precision: 0.7111111111111111
Recall   : 0.6597938144329897
F1 Score : 0.6844919786096256
ROC AUC  : 0.7818036232609349
Decision Tree
Accuracy : 0.62
Precision: 0.6129032258064516
Recall   : 0.5876288659793815
F1 Score : 0.6
ROC AUC  : 0.6190571514362927
Random Forest
Accuracy : 0.725
Precision: 0.7019230769230769
Recall   : 0.7525773195876289
F1 Score : 0.7263681592039801
ROC AUC  : 0.7886097487738964
SVM
Accuracy : 0.73
Precision: 0.6936936936936937
Recall   : 0.7938144329896907
F1 Score : 0.7403846153846154
ROC AUC  : 0.7764988489640676


### Since SVM is giving best score , doing gridsearch to find the best parameter

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", 0.1, 0.01, 0.001]
}

grid = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best CV Score: 0.7073920484736511


In [15]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.675
Precision: 0.6481481481481481
Recall   : 0.7216494845360825
F1 Score : 0.6829268292682927
ROC AUC  : 0.7511760584526073


### Since gridsearch donot increase accuracy so using basemodel as main model .